# How LLMs Work (From the Outside)

Notebook for **Chapter 3** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch03.html

Tip: change a number, predict the output, then run the cell again.

## 3.1 Pretraining: Next-Token Prediction

## 3.2 Tokenization and BPE

In [ ]:
from collections import Counter
corpus={"low":5,"lower":2,"newest":6,"widest":3}
words_={tuple(w)+("</w>",):c for w,c in corpus.items()}
for step in range(6):
    pairs=Counter()
    for w,c in words_.items():
        for a,b in zip(w,w[1:]): pairs[(a,b)]+=c
    best=max(pairs,key=lambda k:(pairs[k],k))
    new={}
    for w,c in words_.items():
        out=[];i=0
        while i<len(w):
            if i<len(w)-1 and (w[i],w[i+1])==best: out.append(w[i]+w[i+1]);i+=2
            else: out.append(w[i]);i+=1
        new[tuple(out)]=c
    words_=new; print(step+1,"merge",best,pairs[best])
print(list(words_))

1 merge ('t', '</w>') 9
2 merge ('s', 't</w>') 9
3 merge ('e', 'st</w>') 9
4 merge ('o', 'w') 7
5 merge ('l', 'ow') 7
6 merge ('w', 'est</w>') 6
[('low', '</w>'), ('low', 'e', 'r', '</w>'), ('n', 'e', 'west</w>'), ('w', 'i', 'd', 'est</w>')]


## 3.3 Sampling: Temperature, Top-k, Top-p

In [ ]:
import numpy as np
logits=np.array([4.0,3.0,2.0,0.5,0.0]); words=["the","a","cat","zebra","xylo"]
def sm(z,T=1.0):
    z=z/T; e=np.exp(z-z.max()); return e/e.sum()
for T in [0.5,1.0,2.0]: print("T",T,np.round(sm(logits,T),4))
p=sm(logits)
idx=np.argsort(-p)[:3]; q=np.zeros(5); q[idx]=p[idx]; print("top-k3",np.round(q/q.sum(),4))
o=np.argsort(-p); c=np.cumsum(p[o]); print("cumsum",np.round(c,4))
keep=o[:np.searchsorted(c,0.9)+1]; q=np.zeros(5); q[keep]=p[keep]
print("top-p0.9 keep",[words[i] for i in keep],np.round(q/q.sum(),4))
rng=np.random.default_rng(0)
print("samples T=0.5",[words[i] for i in rng.choice(5,10,p=sm(logits,0.5))])
print("samples T=2.0",[words[i] for i in rng.choice(5,10,p=sm(logits,2.0))])

T 0.5 [8.659e-01 1.172e-01 1.590e-02 8.000e-04 3.000e-04]
T 1.0 [0.6444 0.2371 0.0872 0.0195 0.0118]
T 2.0 [0.4379 0.2656 0.1611 0.0761 0.0593]
top-k3 [0.6652 0.2447 0.09   0.     0.    ]
cumsum [0.6444 0.8815 0.9687 0.9882 1.    ]
top-p0.9 keep ['the', 'a', 'cat'] [0.6652 0.2447 0.09   0.     0.    ]
samples T=0.5 ['the', 'the', 'the', 'the', 'the', 'a', 'the', 'the', 'the', 'a']
samples T=2.0 ['cat', 'the', 'cat', 'the', 'cat', 'the', 'cat', 'a', 'the', 'the']


## 3.4 Post-Training: SFT, RLHF, DPO, and Reasoning Models

## 3.5 Inference Engineering and Limitations

## 3.6 Attention Cost, the KV Cache and Serving Math

In [ ]:
def kv_bytes_per_token(layers, kv_heads, head_dim, nbytes=2):
    return 2 * layers * kv_heads * head_dim * nbytes

b7 = kv_bytes_per_token(32, 32, 128)
print("7B bytes per token:", b7)
print("7B, 4096 tokens, GB:", round(b7 * 4096 / 1e9, 2))
b70 = kv_bytes_per_token(80, 8, 128)
print("70B GQA bytes per token:", b70)
print("70B GQA, 128k tokens, GB:", round(b70 * 131072 / 1e9, 1))
print("70B if it had 64 KV heads, 128k tokens, GB:", round(kv_bytes_per_token(80, 64, 128) * 131072 / 1e9, 1))

7B bytes per token: 524288
7B, 4096 tokens, GB: 2.15
70B GQA bytes per token: 327680
70B GQA, 128k tokens, GB: 42.9
70B if it had 64 KV heads, 128k tokens, GB: 343.6


In [ ]:
usable = 80 * 0.9 - 14
for name, heads in [("MHA 32", 32), ("GQA 8", 8)]:
    per = 2 * 32 * heads * 128 * 2 * 4096 / 1e9
    print(name, "cache/conv GB:", round(per, 3), "conversations:", int(usable // per))

MHA 32 cache/conv GB: 2.147 conversations: 27
GQA 8 cache/conv GB: 0.537 conversations: 108


In [ ]:
bw, flops = 2.0e12, 312e12
weights = 7e9 * 2
print("batch-1 decode tokens/s:", round(bw / weights))
print("ops per byte the GPU can do:", round(flops / bw))
for batch in (1, 8, 32, 128):
    print("batch", batch, "-> aggregate tokens/s ~", round(batch * bw / weights))

batch-1 decode tokens/s: 143
ops per byte the GPU can do: 156
batch 1 -> aggregate tokens/s ~ 143
batch 8 -> aggregate tokens/s ~ 1143
batch 32 -> aggregate tokens/s ~ 4571
batch 128 -> aggregate tokens/s ~ 18286


In [ ]:
def tokens_per_step(a, k): return (1 - a ** (k + 1)) / (1 - a)
print(round(tokens_per_step(0.8, 4), 2))
print(round(tokens_per_step(0.5, 4), 2))
print(round(tokens_per_step(0.9, 8), 2))

3.36
1.94
6.13


## Chapter recap

## Practice set

Problem 3.P.1 (easy) Logits `[2, 0]`, T = 1. Probability of the first token? **Answer:** e² = 7.389, e⁰ = 1; 7.389 / 8.389 = 0.881.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch03.html